| | |
|---|---|
| **Project** | Advance RAG 29 - Multilingual RAG Bot |
| **Topic** | Cross-lingual retrieval and answer generation |
| **Stack** | sentence-transformers (paraphrase-multilingual-MiniLM-L12-v2), NumPy |
| **Focus** | Query in one language, retrieve and answer in another |

## Overview

Traditional RAG systems work within a single language. A **Multilingual RAG Bot** breaks this barrier:
a user can ask a question in English and retrieve relevant documents originally written in French,
Spanish, German, or other languages -- and vice versa.

This notebook builds a fully self-contained multilingual RAG system that:
1. Embeds documents written in 6 different languages into a shared vector space
2. Performs cross-lingual retrieval (query language != document language)
3. Generates answers that synthesize information across languages
4. Evaluates retrieval quality across language pairs

The key enabler is **multilingual sentence embeddings** -- models trained on parallel corpora
so that semantically equivalent sentences in different languages map to nearby points in the
same embedding space.

## Learning Goals

1. Understand how multilingual embedding models create a shared semantic space across languages
2. Build a cross-lingual retrieval pipeline where query and document languages differ
3. Implement language-aware answer generation that responds in the user's language
4. Evaluate retrieval quality across language pairs with quantitative metrics
5. Analyze failure modes of multilingual embeddings (semantic drift, cultural context gaps)

## Problem Statement

A global knowledge base contains documents in multiple languages. Users should be able to:
- Ask questions in their preferred language
- Retrieve relevant information regardless of the source document's language
- Receive answers in the language they queried in

Standard monolingual embeddings fail here because English queries cannot match French documents.
We need embeddings that map semantically equivalent text to similar vectors across languages.

## Why Multilingual RAG Matters

- **Global enterprises** have documentation in dozens of languages
- **Research** is published in many languages -- English-only retrieval misses critical work
- **Government and NGOs** operate across language boundaries
- **Customer support** must serve users in their native language regardless of KB language
- Without multilingual RAG, organizations must translate everything or accept information silos

## Environment Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'sentence-transformers', 'numpy'])
print('Dependencies ready.')

## Imports

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from collections import defaultdict
import time
import warnings
warnings.filterwarnings('ignore')
print('Imports complete.')

## Configuration

In [ ]:
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
TOP_K = 3
SIMILARITY_THRESHOLD = 0.25

# Language codes used in the corpus
LANG_NAMES = {
    'en': 'English',
    'fr': 'French',
    'es': 'Spanish',
    'de': 'German',
    'pt': 'Portuguese',
    'ja': 'Japanese'
}
print(f'Model: {MODEL_NAME}')
print(f'Languages: {list(LANG_NAMES.values())}')

## Multilingual Embedding Model

**paraphrase-multilingual-MiniLM-L12-v2** is trained on parallel corpora spanning 50+ languages.
It maps semantically equivalent sentences to nearby vectors regardless of language.

How it works:
- Training uses translation pairs: (English sentence, French translation) should have similar embeddings
- Knowledge distillation from a larger teacher model into a compact 12-layer student
- The result: "The cat sits on the mat" and "Le chat est assis sur le tapis" get nearly identical vectors

This shared space is what enables cross-lingual retrieval.

In [ ]:
t0 = time.time()
model = SentenceTransformer(MODEL_NAME)
t1 = time.time()
print(f'Model loaded in {t1 - t0:.1f}s')
print(f'Embedding dimension: {model.get_sentence_embedding_dimension()}')

## Multilingual Knowledge Base

We create a corpus of 36 documents across 6 languages covering 6 topics.
Each topic has one document per language expressing equivalent content,
allowing us to test whether cross-lingual retrieval finds the right matches.

In [ ]:
# 36 documents: 6 topics x 6 languages
# Each topic has semantically equivalent content in each language
documents = [
    # Topic 1: Climate Change
    {'id': 'CC-EN', 'lang': 'en', 'topic': 'climate',
     'text': 'Climate change is causing global temperatures to rise. The Paris Agreement aims to limit warming to 1.5 degrees Celsius. Renewable energy adoption is critical to reducing greenhouse gas emissions.'},
    {'id': 'CC-FR', 'lang': 'fr', 'topic': 'climate',
     'text': 'Le changement climatique provoque une hausse des temperatures mondiales. L\'Accord de Paris vise a limiter le rechauffement a 1,5 degre Celsius. L\'adoption des energies renouvelables est essentielle pour reduire les emissions de gaz a effet de serre.'},
    {'id': 'CC-ES', 'lang': 'es', 'topic': 'climate',
     'text': 'El cambio climatico esta provocando un aumento de las temperaturas globales. El Acuerdo de Paris busca limitar el calentamiento a 1,5 grados Celsius. La adopcion de energias renovables es fundamental para reducir las emisiones de gases de efecto invernadero.'},
    {'id': 'CC-DE', 'lang': 'de', 'topic': 'climate',
     'text': 'Der Klimawandel verursacht einen Anstieg der globalen Temperaturen. Das Pariser Abkommen zielt darauf ab, die Erwaermung auf 1,5 Grad Celsius zu begrenzen. Die Nutzung erneuerbarer Energien ist entscheidend fuer die Reduzierung der Treibhausgasemissionen.'},
    {'id': 'CC-PT', 'lang': 'pt', 'topic': 'climate',
     'text': 'As mudancas climaticas estao causando o aumento das temperaturas globais. O Acordo de Paris visa limitar o aquecimento a 1,5 graus Celsius. A adocao de energias renovaveis e fundamental para reduzir as emissoes de gases de efeito estufa.'},
    {'id': 'CC-JA', 'lang': 'ja', 'topic': 'climate',
     'text': '気候変動により世界の気温が上昇しています。パリ協定は温暖化を1.5度に抑えることを目指しています。温室効果ガスの排出削減には再生可能エネルギーの導入が不可欠です。'},

    # Topic 2: Artificial Intelligence
    {'id': 'AI-EN', 'lang': 'en', 'topic': 'ai',
     'text': 'Artificial intelligence is transforming industries from healthcare to finance. Machine learning models can now diagnose diseases, drive cars, and translate languages with remarkable accuracy.'},
    {'id': 'AI-FR', 'lang': 'fr', 'topic': 'ai',
     'text': 'L\'intelligence artificielle transforme les industries, de la sante a la finance. Les modeles d\'apprentissage automatique peuvent desormais diagnostiquer des maladies, conduire des voitures et traduire des langues avec une precision remarquable.'},
    {'id': 'AI-ES', 'lang': 'es', 'topic': 'ai',
     'text': 'La inteligencia artificial esta transformando industrias desde la salud hasta las finanzas. Los modelos de aprendizaje automatico ahora pueden diagnosticar enfermedades, conducir autos y traducir idiomas con notable precision.'},
    {'id': 'AI-DE', 'lang': 'de', 'topic': 'ai',
     'text': 'Kuenstliche Intelligenz veraendert Branchen von Gesundheitswesen bis Finanzwirtschaft. Modelle des maschinellen Lernens koennen heute Krankheiten diagnostizieren, Autos fahren und Sprachen mit bemerkenswerter Genauigkeit uebersetzen.'},
    {'id': 'AI-PT', 'lang': 'pt', 'topic': 'ai',
     'text': 'A inteligencia artificial esta transformando industrias da saude as financas. Modelos de aprendizado de maquina agora podem diagnosticar doencas, dirigir carros e traduzir idiomas com notavel precisao.'},
    {'id': 'AI-JA', 'lang': 'ja', 'topic': 'ai',
     'text': '人工知能は医療から金融まで産業を変革しています。機械学習モデルは病気の診断、自動車の運転、言語の翻訳を驚くべき精度で行えるようになりました。'},

    # Topic 3: Space Exploration
    {'id': 'SP-EN', 'lang': 'en', 'topic': 'space',
     'text': 'NASA and SpaceX are planning crewed missions to Mars by the 2030s. The Artemis program has returned humans to the Moon. Private space companies are making orbital flight more accessible.'},
    {'id': 'SP-FR', 'lang': 'fr', 'topic': 'space',
     'text': 'La NASA et SpaceX preparent des missions habitees vers Mars d\'ici les annees 2030. Le programme Artemis a ramene des humains sur la Lune. Les entreprises spatiales privees rendent le vol orbital plus accessible.'},
    {'id': 'SP-ES', 'lang': 'es', 'topic': 'space',
     'text': 'La NASA y SpaceX estan planificando misiones tripuladas a Marte para la decada de 2030. El programa Artemis ha regresado humanos a la Luna. Las empresas espaciales privadas hacen el vuelo orbital mas accesible.'},
    {'id': 'SP-DE', 'lang': 'de', 'topic': 'space',
     'text': 'Die NASA und SpaceX planen bemannte Missionen zum Mars bis in die 2030er Jahre. Das Artemis-Programm hat Menschen zurueck zum Mond gebracht. Private Raumfahrtunternehmen machen die Orbitalfahrt zugaenglicher.'},
    {'id': 'SP-PT', 'lang': 'pt', 'topic': 'space',
     'text': 'A NASA e a SpaceX estao planejando missoes tripuladas a Marte ate a decada de 2030. O programa Artemis retornou humanos a Lua. Empresas espaciais privadas estao tornando o voo orbital mais acessivel.'},
    {'id': 'SP-JA', 'lang': 'ja', 'topic': 'space',
     'text': 'NASAとSpaceXは2030年代までに火星への有人ミッションを計画しています。アルテミス計画は人類を月に帰還させました。民間宇宙企業が軌道飛行をより身近にしています。'},

    # Topic 4: Quantum Computing
    {'id': 'QC-EN', 'lang': 'en', 'topic': 'quantum',
     'text': 'Quantum computers use qubits that can exist in superposition, enabling parallel computation. Google and IBM are racing to achieve practical quantum advantage for real-world problems.'},
    {'id': 'QC-FR', 'lang': 'fr', 'topic': 'quantum',
     'text': 'Les ordinateurs quantiques utilisent des qubits pouvant exister en superposition, permettant le calcul parallele. Google et IBM se disputent l\'avantage quantique pratique pour des problemes reels.'},
    {'id': 'QC-ES', 'lang': 'es', 'topic': 'quantum',
     'text': 'Las computadoras cuanticas usan qubits que pueden existir en superposicion, lo que permite la computacion paralela. Google e IBM compiten por lograr una ventaja cuantica practica para problemas reales.'},
    {'id': 'QC-DE', 'lang': 'de', 'topic': 'quantum',
     'text': 'Quantencomputer verwenden Qubits, die in Superposition existieren koennen und parallele Berechnungen ermoeglichen. Google und IBM wetteifern um einen praktischen Quantenvorteil fuer reale Probleme.'},
    {'id': 'QC-PT', 'lang': 'pt', 'topic': 'quantum',
     'text': 'Computadores quanticos usam qubits que podem existir em superposicao, permitindo computacao paralela. Google e IBM disputam a vantagem quantica pratica para problemas reais.'},
    {'id': 'QC-JA', 'lang': 'ja', 'topic': 'quantum',
     'text': '量子コンピュータは重ね合わせ状態を持つ量子ビットを使い並列計算を可能にします。GoogleとIBMは実用的な量子優位性の実現を競っています。'},

    # Topic 5: Global Health
    {'id': 'GH-EN', 'lang': 'en', 'topic': 'health',
     'text': 'The COVID-19 pandemic accelerated vaccine technology, particularly mRNA platforms. Global health systems are now focused on pandemic preparedness and equitable vaccine distribution worldwide.'},
    {'id': 'GH-FR', 'lang': 'fr', 'topic': 'health',
     'text': 'La pandemie de COVID-19 a accelere la technologie vaccinale, notamment les plateformes ARNm. Les systemes de sante mondiaux se concentrent desormais sur la preparedness pandemique et la distribution equitable des vaccins.'},
    {'id': 'GH-ES', 'lang': 'es', 'topic': 'health',
     'text': 'La pandemia de COVID-19 acelero la tecnologia de vacunas, particularmente las plataformas de ARNm. Los sistemas de salud global se centran ahora en la preparacion para pandemias y la distribucion equitativa de vacunas.'},
    {'id': 'GH-DE', 'lang': 'de', 'topic': 'health',
     'text': 'Die COVID-19-Pandemie beschleunigte die Impfstofftechnologie, insbesondere mRNA-Plattformen. Globale Gesundheitssysteme konzentrieren sich nun auf Pandemievorsorge und gerechte Impfstoffverteilung weltweit.'},
    {'id': 'GH-PT', 'lang': 'pt', 'topic': 'health',
     'text': 'A pandemia de COVID-19 acelerou a tecnologia de vacinas, particularmente plataformas de mRNA. Os sistemas de saude global agora focam na preparacao para pandemias e distribuicao equitativa de vacinas.'},
    {'id': 'GH-JA', 'lang': 'ja', 'topic': 'health',
     'text': 'COVID-19パンデミックはワクチン技術、特にmRNAプラットフォームの開発を加速させました。世界の保健システムはパンデミックへの備えとワクチンの公平な分配に注力しています。'},

    # Topic 6: Renewable Energy
    {'id': 'RE-EN', 'lang': 'en', 'topic': 'energy',
     'text': 'Solar and wind energy now account for a growing share of global electricity generation. Battery storage technology is advancing rapidly, addressing the intermittency challenge of renewables.'},
    {'id': 'RE-FR', 'lang': 'fr', 'topic': 'energy',
     'text': 'L\'energie solaire et eolienne represente une part croissante de la production mondiale d\'electricite. La technologie de stockage par batterie progresse rapidement, repondant au defi de l\'intermittence des renouvelables.'},
    {'id': 'RE-ES', 'lang': 'es', 'topic': 'energy',
     'text': 'La energia solar y eolica representan una parte creciente de la generacion electrica mundial. La tecnologia de almacenamiento en baterias avanza rapidamente, abordando el desafio de la intermitencia de las renovables.'},
    {'id': 'RE-DE', 'lang': 'de', 'topic': 'energy',
     'text': 'Solar- und Windenergie machen einen wachsenden Anteil der globalen Stromerzeugung aus. Batteriespeichertechnologie entwickelt sich rasant und adressiert die Intermittenz-Herausforderung erneuerbarer Energien.'},
    {'id': 'RE-PT', 'lang': 'pt', 'topic': 'energy',
     'text': 'A energia solar e eolica agora representa uma parcela crescente da geracao global de eletricidade. A tecnologia de armazenamento em baterias avanca rapidamente, enfrentando o desafio da intermitencia das renovaveis.'},
    {'id': 'RE-JA', 'lang': 'ja', 'topic': 'energy',
     'text': '太陽光発電と風力発電は世界の発電量に占める割合が増加しています。バッテリー蓄電技術は急速に進歩し再生可能エネルギーの断続性の課題に対応しています。'},
]

print(f'Corpus size: {len(documents)} documents')
print(f'Topics: {sorted(set(d["topic"] for d in documents))}')
print(f'Languages: {sorted(set(d["lang"] for d in documents))}')

# Verify balanced corpus
for topic in sorted(set(d['topic'] for d in documents)):
    langs = sorted([d['lang'] for d in documents if d['topic'] == topic])
    print(f'  {topic}: {langs}')

## Embed All Documents

In [ ]:
texts = [d['text'] for d in documents]
t0 = time.time()
doc_embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
t1 = time.time()
print(f'Encoded {len(texts)} documents in {t1-t0:.2f}s')
print(f'Embedding shape: {doc_embeddings.shape}')

## Cross-Lingual Similarity Demonstration

Before building retrieval, let us verify that the multilingual model maps equivalent
content in different languages to similar vectors. We compute cosine similarity between
all pairs of documents on the same topic but in different languages.

In [ ]:
# Within-topic cross-lingual similarity
topics = sorted(set(d['topic'] for d in documents))
print('Cross-lingual similarity within each topic:')
print('=' * 65)

topic_similarities = {}
for topic in topics:
    idxs = [i for i, d in enumerate(documents) if d['topic'] == topic]
    sims = []
    for a in range(len(idxs)):
        for b in range(a+1, len(idxs)):
            sim = float(np.dot(doc_embeddings[idxs[a]], doc_embeddings[idxs[b]]))
            sims.append(sim)
    avg_sim = float(np.mean(sims))
    min_sim = float(np.min(sims))
    topic_similarities[topic] = avg_sim
    print(f'  {topic:10s}: avg={avg_sim:.4f}  min={min_sim:.4f}  (across {len(sims)} language pairs)')

overall_avg = float(np.mean(list(topic_similarities.values())))
print(f'\nOverall avg within-topic similarity: {overall_avg:.4f}')

## Cross-Topic Similarity (Negative Control)

To confirm the model discriminates topics, we compare documents from different topics.
Cross-topic similarity should be significantly lower than within-topic similarity.

In [ ]:
cross_topic_sims = []
for i in range(len(documents)):
    for j in range(i+1, len(documents)):
        if documents[i]['topic'] != documents[j]['topic']:
            sim = float(np.dot(doc_embeddings[i], doc_embeddings[j]))
            cross_topic_sims.append(sim)

cross_avg = float(np.mean(cross_topic_sims))
cross_max = float(np.max(cross_topic_sims))
print(f'Cross-topic similarity: avg={cross_avg:.4f}  max={cross_max:.4f}')
print(f'Within-topic similarity: avg={overall_avg:.4f}')
gap = overall_avg - cross_avg
print(f'Discrimination gap: {gap:.4f}')
if gap > 0.15:
    print('Good topic discrimination.')
else:
    print('Warning: low topic discrimination.')

## Cross-Lingual Retrieval Engine

The retrieval engine encodes the query with the same multilingual model and finds
the most similar documents by cosine similarity -- regardless of language.

In [ ]:
def retrieve(query, top_k=TOP_K, exclude_lang=None):
    """Retrieve top-k documents for a query.
    Optionally exclude documents in a specific language (for cross-lingual tests)."""
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    sims = np.dot(doc_embeddings, q_emb)
    
    results = []
    for i in np.argsort(sims)[::-1]:
        doc = documents[i]
        if exclude_lang and doc['lang'] == exclude_lang:
            continue
        results.append({
            'id': doc['id'],
            'lang': doc['lang'],
            'topic': doc['topic'],
            'text': doc['text'],
            'score': float(sims[i])
        })
        if len(results) >= top_k:
            break
    return results

# Quick test
test_results = retrieve('What is climate change?')
print('Test query: "What is climate change?"')
for r in test_results:
    print(f'  [{r["id"]}] lang={r["lang"]} topic={r["topic"]} score={r["score"]:.4f}')

## Language Detection

To generate answers in the user's query language, we need to detect what language
the query is in. We use a simple embedding similarity approach: compare the query
against reference phrases in each supported language.

In [ ]:
# Reference phrases for language detection
LANG_REFS = {
    'en': 'This is a question written in the English language.',
    'fr': 'Ceci est une question ecrite en langue francaise.',
    'es': 'Esta es una pregunta escrita en idioma espanol.',
    'de': 'Dies ist eine Frage in deutscher Sprache geschrieben.',
    'pt': 'Esta e uma pergunta escrita em lingua portuguesa.',
    'ja': 'これは日本語で書かれた質問です。'
}

ref_texts = list(LANG_REFS.values())
ref_langs = list(LANG_REFS.keys())
ref_embeddings = model.encode(ref_texts, normalize_embeddings=True)

def detect_language(text):
    """Detect the language of input text using embedding similarity to reference phrases."""
    q_emb = model.encode([text], normalize_embeddings=True)[0]
    sims = np.dot(ref_embeddings, q_emb)
    best_idx = int(np.argmax(sims))
    return ref_langs[best_idx], float(sims[best_idx])

# Test language detection
test_queries = [
    'What are the effects of climate change?',
    'Quels sont les effets du changement climatique?',
    'Cuales son los efectos del cambio climatico?',
    'Was sind die Auswirkungen des Klimawandels?',
    'Quais sao os efeitos das mudancas climaticas?',
    '気候変動の影響は何ですか？'
]

print('Language detection test:')
lang_correct = 0
expected_langs = ['en', 'fr', 'es', 'de', 'pt', 'ja']
for q, exp in zip(test_queries, expected_langs):
    detected, conf = detect_language(q)
    match = 'OK' if detected == exp else 'MISS'
    if detected == exp:
        lang_correct += 1
    print(f'  [{match}] "{q[:50]}..." -> {detected} (conf={conf:.3f}, expected={exp})')
print(f'Language detection accuracy: {lang_correct}/{len(expected_langs)}')

## Answer Generation Pipeline

Since we do not use an LLM, we build a rule-based answer generator that:
1. Detects the query language
2. Retrieves relevant documents (any language)
3. Synthesizes an answer from retrieved content
4. Wraps the answer with language-appropriate framing

In production, this synthesis step would be handled by a multilingual LLM (e.g., Qwen3-Instruct).
Here we demonstrate the retrieval mechanics and present retrieved content.

In [ ]:
ANSWER_TEMPLATES = {
    'en': 'Based on {n} retrieved documents (from {langs}), here is the answer:\n\n{content}',
    'fr': 'Sur la base de {n} documents recuperes (en {langs}), voici la reponse:\n\n{content}',
    'es': 'Basado en {n} documentos recuperados (en {langs}), aqui esta la respuesta:\n\n{content}',
    'de': 'Basierend auf {n} abgerufenen Dokumenten (in {langs}), hier ist die Antwort:\n\n{content}',
    'pt': 'Com base em {n} documentos recuperados (em {langs}), aqui esta a resposta:\n\n{content}',
    'ja': '{n}件の取得文書（{langs}）に基づいた回答：\n\n{content}'
}

def generate_answer(query, top_k=TOP_K, force_lang=None):
    """Full multilingual RAG pipeline: detect language, retrieve, generate answer."""
    # Step 1: Detect query language
    query_lang, conf = detect_language(query)
    answer_lang = force_lang if force_lang else query_lang
    
    # Step 2: Retrieve documents (all languages)
    results = retrieve(query, top_k=top_k)
    
    # Step 3: Assemble answer
    source_langs = sorted(set(r['lang'] for r in results))
    lang_labels = ', '.join(LANG_NAMES.get(l, l) for l in source_langs)
    
    content_parts = []
    for i, r in enumerate(results):
        content_parts.append(f'[{r["id"]}] ({LANG_NAMES[r["lang"]]}, score={r["score"]:.3f}):\n{r["text"]}')
    content = '\n\n'.join(content_parts)
    
    template = ANSWER_TEMPLATES.get(answer_lang, ANSWER_TEMPLATES['en'])
    answer = template.format(n=len(results), langs=lang_labels, content=content)
    
    return {
        'query': query,
        'query_lang': query_lang,
        'answer_lang': answer_lang,
        'answer': answer,
        'results': results,
        'source_langs': source_langs
    }

print('Answer generation pipeline ready.')

## Cross-Lingual Test Queries

We test the system with queries in different languages. The key test is whether
queries retrieve relevant documents from OTHER languages, not just same-language matches.

In [ ]:
test_cases = [
    # Same-language retrieval (baseline)
    {'query': 'How does artificial intelligence transform healthcare?',
     'expected_topic': 'ai', 'query_lang': 'en', 'type': 'same-lang'},
    
    # Cross-lingual: French query -> should retrieve from all langs about AI
    {'query': 'Comment l\'intelligence artificielle transforme-t-elle la sante?',
     'expected_topic': 'ai', 'query_lang': 'fr', 'type': 'cross-lingual'},
    
    # Cross-lingual: Spanish query about space
    {'query': 'Cuando llegaran los humanos a Marte?',
     'expected_topic': 'space', 'query_lang': 'es', 'type': 'cross-lingual'},
    
    # Cross-lingual: German query about quantum computing
    {'query': 'Was sind Quantencomputer und wie funktionieren Qubits?',
     'expected_topic': 'quantum', 'query_lang': 'de', 'type': 'cross-lingual'},
    
    # Cross-lingual: Portuguese query about health
    {'query': 'Como a pandemia acelerou o desenvolvimento de vacinas?',
     'expected_topic': 'health', 'query_lang': 'pt', 'type': 'cross-lingual'},
    
    # Cross-lingual: Japanese query about energy
    {'query': '再生可能エネルギーの課題は何ですか？',
     'expected_topic': 'energy', 'query_lang': 'ja', 'type': 'cross-lingual'},
    
    # Cross-lingual: English query, force answer in French
    {'query': 'Tell me about renewable energy and battery storage',
     'expected_topic': 'energy', 'query_lang': 'en', 'type': 'forced-lang',
     'force_lang': 'fr'},
    
    # Cross-lingual: French query, force answer in German
    {'query': 'Parlez-moi de l\'exploration spatiale et de Mars',
     'expected_topic': 'space', 'query_lang': 'fr', 'type': 'forced-lang',
     'force_lang': 'de'},
]

print(f'Test cases: {len(test_cases)}')
for i, tc in enumerate(test_cases):
    print(f'  Q{i+1} [{tc["type"]:14s}] ({tc["query_lang"]}) {tc["query"][:60]}...')

## Run Full Benchmark

In [ ]:
all_results = []
print('Running multilingual RAG benchmark...')
print('=' * 80)

for i, tc in enumerate(test_cases):
    force = tc.get('force_lang', None)
    out = generate_answer(tc['query'], force_lang=force)
    
    # Check: did retrieval hit the expected topic?
    top1_topic = out['results'][0]['topic'] if out['results'] else None
    topk_topics = [r['topic'] for r in out['results']]
    hit_at_1 = top1_topic == tc['expected_topic']
    hit_at_k = tc['expected_topic'] in topk_topics
    
    # Check: did retrieval find docs in OTHER languages?
    retrieved_langs = [r['lang'] for r in out['results']]
    cross_lingual_count = sum(1 for l in retrieved_langs if l != tc['query_lang'])
    
    result_row = {
        'qid': f'Q{i+1}',
        'type': tc['type'],
        'query_lang': tc['query_lang'],
        'answer_lang': out['answer_lang'],
        'expected_topic': tc['expected_topic'],
        'hit_at_1': hit_at_1,
        'hit_at_k': hit_at_k,
        'top1_id': out['results'][0]['id'] if out['results'] else 'N/A',
        'top1_score': out['results'][0]['score'] if out['results'] else 0.0,
        'retrieved_langs': retrieved_langs,
        'cross_lingual_count': cross_lingual_count,
        'answer': out['answer']
    }
    all_results.append(result_row)
    
    status = 'HIT' if hit_at_1 else ('PARTIAL' if hit_at_k else 'MISS')
    print(f'\nQ{i+1} [{status}] ({tc["query_lang"]}->{out["answer_lang"]}) "{tc["query"][:55]}..."')
    print(f'  Expected topic: {tc["expected_topic"]}')
    for r in out['results']:
        marker = '*' if r['topic'] == tc['expected_topic'] else ' '
        print(f'  {marker} [{r["id"]}] lang={r["lang"]} topic={r["topic"]} score={r["score"]:.4f}')
    print(f'  Cross-lingual docs retrieved: {cross_lingual_count}/{len(retrieved_langs)}')

print('\n' + '=' * 80)
print('Benchmark complete.')

## Aggregate Metrics

In [ ]:
hit1_count = sum(1 for r in all_results if r['hit_at_1'])
hitk_count = sum(1 for r in all_results if r['hit_at_k'])
n_queries = len(all_results)

hit1_rate = hit1_count / n_queries
hitk_rate = hitk_count / n_queries

avg_top1_score = float(np.mean([r['top1_score'] for r in all_results]))
avg_cross_lingual = float(np.mean([r['cross_lingual_count'] for r in all_results]))

print('Overall Multilingual RAG Metrics')
print('=' * 45)
print(f'Hit@1 (topic match):       {hit1_count}/{n_queries} = {hit1_rate:.1%}')
print(f'Hit@{TOP_K} (topic match):       {hitk_count}/{n_queries} = {hitk_rate:.1%}')
print(f'Avg top-1 similarity:      {avg_top1_score:.4f}')
print(f'Avg cross-lingual docs:    {avg_cross_lingual:.1f}/{TOP_K}')

# Per query-type breakdown
print('\nPer query type:')
for qtype in ['same-lang', 'cross-lingual', 'forced-lang']:
    subset = [r for r in all_results if r['type'] == qtype]
    if not subset:
        continue
    h1 = sum(1 for r in subset if r['hit_at_1'])
    hk = sum(1 for r in subset if r['hit_at_k'])
    cl = float(np.mean([r['cross_lingual_count'] for r in subset]))
    print(f'  {qtype:14s}: Hit@1={h1}/{len(subset)}  Hit@K={hk}/{len(subset)}  avg_cross_lingual={cl:.1f}')

## Cross-Lingual Retrieval Matrix

We systematically test retrieval across all language pairs on the 'climate' topic.
For each query language, we exclude same-language documents and measure whether
the correct topic is retrieved from other languages.

In [ ]:
# Use climate topic as test case
climate_queries = {
    'en': 'What is climate change and the Paris Agreement?',
    'fr': 'Qu\'est-ce que le changement climatique et l\'Accord de Paris?',
    'es': 'Que es el cambio climatico y el Acuerdo de Paris?',
    'de': 'Was ist der Klimawandel und das Pariser Abkommen?',
    'pt': 'O que e mudanca climatica e o Acordo de Paris?',
    'ja': '気候変動とパリ協定とは何ですか？'
}

print('Cross-lingual retrieval matrix (climate topic, K=3, same-lang excluded):')
print(f'{"Query":>8s} | {"Retrieved Languages":30s} | {"Top1 Topic":10s} | Score')
print('-' * 70)

matrix_hits = 0
matrix_total = 0
for qlang, query in climate_queries.items():
    results = retrieve(query, top_k=3, exclude_lang=qlang)
    r_langs = [r['lang'] for r in results]
    top_topic = results[0]['topic'] if results else 'N/A'
    top_score = results[0]['score'] if results else 0.0
    hit = top_topic == 'climate'
    matrix_hits += int(hit)
    matrix_total += 1
    marker = 'OK' if hit else 'MISS'
    print(f'{qlang:>8s} | {str(r_langs):30s} | {top_topic:10s} | {top_score:.4f} [{marker}]')

print(f'\nCross-lingual Hit@1: {matrix_hits}/{matrix_total} = {matrix_hits/matrix_total:.1%}')

## Language Pair Similarity Analysis

We compute average embedding similarity between each language pair across all topics
to understand which language pairs the model handles best.

In [ ]:
lang_list = sorted(set(d['lang'] for d in documents))
n_langs = len(lang_list)
pair_sims = np.zeros((n_langs, n_langs))

for topic in topics:
    topic_docs = [(i, d) for i, d in enumerate(documents) if d['topic'] == topic]
    for a_idx in range(len(topic_docs)):
        for b_idx in range(len(topic_docs)):
            i_a, d_a = topic_docs[a_idx]
            i_b, d_b = topic_docs[b_idx]
            li = lang_list.index(d_a['lang'])
            lj = lang_list.index(d_b['lang'])
            sim = float(np.dot(doc_embeddings[i_a], doc_embeddings[i_b]))
            pair_sims[li][lj] += sim

# Average over topics
pair_sims /= len(topics)

print('Average within-topic similarity by language pair:')
header = f'{"":>4s} | ' + ' | '.join(f'{l:>5s}' for l in lang_list)
print(header)
print('-' * len(header))
for i, l1 in enumerate(lang_list):
    row = f'{l1:>4s} | ' + ' | '.join(f'{pair_sims[i][j]:.3f}' for j in range(n_langs))
    print(row)

# Find best and worst pairs (excluding diagonal)
pairs = []
for i in range(n_langs):
    for j in range(i+1, n_langs):
        pairs.append((lang_list[i], lang_list[j], pair_sims[i][j]))
pairs.sort(key=lambda x: x[2], reverse=True)

print(f'\nBest language pair:  {pairs[0][0]}-{pairs[0][1]} ({pairs[0][2]:.4f})')
print(f'Worst language pair: {pairs[-1][0]}-{pairs[-1][1]} ({pairs[-1][2]:.4f})')

## Per-Query Detailed Breakdown

In [ ]:
print(f'{"QID":>4s} | {"Type":>14s} | {"QLang":>5s} | {"ALang":>5s} | {"Topic":>8s} | {"Hit@1":>5s} | {"Hit@K":>5s} | {"XLing":>5s} | {"Top1":>6s}')
print('-' * 90)
for r in all_results:
    print(f'{r["qid"]:>4s} | {r["type"]:>14s} | {r["query_lang"]:>5s} | {r["answer_lang"]:>5s} | '
          f'{r["expected_topic"]:>8s} | {str(r["hit_at_1"]):>5s} | {str(r["hit_at_k"]):>5s} | '
          f'{r["cross_lingual_count"]:>5d} | {r["top1_score"]:>6.4f}')

## Qualitative Comparison: Same-Language vs Cross-Lingual

We compare retrieval results when the query language matches the document language
versus when it does not, to see how well cross-lingual retrieval preserves relevance.

In [ ]:
# Same-language: English query, include English docs
print('=== Same-Language Retrieval ===')
print('Query (en): "How does AI transform healthcare?"')
same_results = retrieve('How does AI transform healthcare?', top_k=3)
for r in same_results:
    print(f'  [{r["id"]}] lang={r["lang"]} score={r["score"]:.4f}')
    print(f'    {r["text"][:100]}...')

print()

# Cross-lingual: English query, exclude English docs
print('=== Cross-Lingual Retrieval (English docs excluded) ===')
print('Query (en): "How does AI transform healthcare?"')
cross_results = retrieve('How does AI transform healthcare?', top_k=3, exclude_lang='en')
for r in cross_results:
    print(f'  [{r["id"]}] lang={r["lang"]} score={r["score"]:.4f}')
    print(f'    {r["text"][:100]}...')

# Score drop analysis
same_top = same_results[0]['score']
cross_top = cross_results[0]['score']
drop = same_top - cross_top
print(f'\nScore drop (same-lang top1 vs cross-lang top1): {drop:.4f} ({drop/same_top:.1%} relative)')

## Full Answer Examples

In [ ]:
# Example 1: French query
print('Example 1: French query about space exploration')
print('=' * 60)
out1 = generate_answer('Quand les humains iront-ils sur Mars?')
print(f'Detected language: {out1["query_lang"]} -> Answer in: {out1["answer_lang"]}')
print(out1['answer'])

print()

# Example 2: Japanese query
print('Example 2: Japanese query about quantum computing')
print('=' * 60)
out2 = generate_answer('量子コンピュータの仕組みを教えてください')
print(f'Detected language: {out2["query_lang"]} -> Answer in: {out2["answer_lang"]}')
print(out2['answer'])

print()

# Example 3: English query, forced German answer
print('Example 3: English query, forced German answer')
print('=' * 60)
out3 = generate_answer('What are the latest developments in vaccine technology?', force_lang='de')
print(f'Detected language: {out3["query_lang"]} -> Answer in: {out3["answer_lang"]}')
print(out3['answer'])

## Error Analysis

In [ ]:
print('Error Analysis')
print('=' * 60)

misses = [r for r in all_results if not r['hit_at_1']]
if misses:
    print(f'\nQueries where top-1 did not match expected topic ({len(misses)}):')
    for r in misses:
        print(f'  {r["qid"]}: expected={r["expected_topic"]}, got top1={r["top1_id"]} (score={r["top1_score"]:.4f})')
        # Diagnose: what topic did we get instead?
        wrong_topic = r['top1_id'].split('-')[0].lower()
        print(f'    Retrieved topic prefix: {wrong_topic}')
        print(f'    Retrieved langs: {r["retrieved_langs"]}')
else:
    print('All queries matched expected topic at rank 1.')

# Analyze language bias: does the model favor certain languages?
print('\nRetrieved language distribution across all queries:')
lang_counts = defaultdict(int)
total_retrieved = 0
for r in all_results:
    for l in r['retrieved_langs']:
        lang_counts[l] += 1
        total_retrieved += 1

for lang in sorted(lang_counts.keys()):
    pct = lang_counts[lang] / total_retrieved * 100
    print(f'  {LANG_NAMES[lang]:12s}: {lang_counts[lang]:3d} ({pct:.1f}%)')

# Check for monolingual bias
same_lang_retrievals = sum(1 for r in all_results for l in r['retrieved_langs'] if l == r['query_lang'])
same_lang_pct = same_lang_retrievals / total_retrieved * 100
print(f'\nSame-language retrievals: {same_lang_retrievals}/{total_retrieved} ({same_lang_pct:.1f}%)')
if same_lang_pct > 50:
    print('Note: model shows some bias toward same-language documents.')
else:
    print('Cross-lingual retrieval is well-distributed.')

## How Multilingual Embeddings Work

### The Core Idea

Multilingual embedding models learn a **shared semantic space** where meaning is preserved
across languages. The key insight: if two sentences mean the same thing in different languages,
they should have nearly identical vector representations.

### Training Process

1. **Parallel corpora**: The model trains on millions of translation pairs
   (e.g., English-French, English-German), learning that translations should map to similar vectors

2. **Knowledge distillation**: A strong English-only teacher model's representations are
   distilled into a multilingual student, transferring semantic understanding to all languages

3. **Cross-lingual alignment**: Additional training aligns the vector spaces so that
   semantically similar content clusters together regardless of language

### Why It Enables Cross-Lingual RAG

- A French query about "intelligence artificielle" maps to the same region as
  English documents about "artificial intelligence"
- Cosine similarity works across languages because the geometric relationships
  are preserved
- No translation step is needed -- the embedding IS the translation

### Limitations

- **Low-resource languages** have weaker alignment (less training data)
- **Cultural concepts** without direct translations may not align well
- **Technical jargon** may be underrepresented in training corpora for some languages
- **Script differences** (Latin vs CJK) can reduce similarity scores slightly

## Structured vs Unstructured Multilingual Retrieval

| Aspect | Monolingual RAG | Multilingual RAG |
|--------|----------------|------------------|
| Embedding model | English-only (e.g., all-MiniLM-L6-v2) | Multilingual (e.g., paraphrase-multilingual-MiniLM-L12-v2) |
| Query-doc language | Must match | Can differ |
| Knowledge coverage | Single language only | All languages in training set |
| Answer generation | Same language as query | Can respond in any supported language |
| Similarity scores | Higher (same-language bias) | Slightly lower cross-lingually |
| Use case | Single-market products | Global enterprises, research, government |

## Limitations

1. **No LLM for answer generation** -- we present retrieved content rather than synthesizing fluent answers
2. **Language detection via embedding similarity** is approximate; a dedicated model (e.g., fastText langdetect) would be more reliable
3. **Parallel corpus design** -- real-world corpora have uneven language coverage, which would affect retrieval balance
4. **Same-language bias** -- multilingual models often score same-language pairs slightly higher, potentially missing relevant foreign-language documents
5. **Japanese script gap** -- CJK languages sometimes show lower cross-lingual similarity due to script and grammar differences
6. **No code-switching handling** -- queries mixing languages (e.g., Spanglish) are not supported

## Common Mistakes

| Mistake | Why It's Wrong | What To Do Instead |
|---------|---------------|--------------------|
| Using English-only embeddings for multilingual content | Cross-lingual retrieval fails completely | Use a multilingual model trained on parallel corpora |
| Translating queries before embedding | Adds latency and translation errors | Embed queries directly in their original language |
| Assuming equal quality across all languages | Low-resource languages have weaker alignment | Evaluate per-language and add language-specific fallbacks |
| Ignoring same-language bias | Model may miss relevant foreign-language docs | Monitor language distribution in results and consider re-ranking |
| Using a single similarity threshold for all pairs | Optimal thresholds vary by language pair | Calibrate thresholds per language pair if possible |

## Mini Challenge

1. Add documents in a 7th language (e.g., Korean or Arabic) and test cross-lingual retrieval quality
2. Implement a re-ranking step that penalizes same-language bias to improve cross-lingual diversity
3. Replace the rule-based answer generator with a multilingual LLM (e.g., Qwen3-Instruct) for fluent cross-lingual answers
4. Build a language-pair confusion matrix showing how often queries in language X retrieve documents in language Y
5. Test with domain-specific queries (medical, legal) and measure whether technical terminology reduces cross-lingual similarity

## Production Considerations

| Consideration | Recommendation |
|--------------|----------------|
| Embedding model | Use larger models (e.g., multilingual-e5-large, BGE-M3) for better cross-lingual alignment |
| Language detection | Use dedicated language detector (fastText, lingua-py) instead of embedding similarity |
| Answer generation | Use Qwen3-Instruct or similar multilingual LLM for fluent cross-lingual answers |
| Indexing | Use FAISS or Milvus with language metadata for filtered retrieval |
| Evaluation | Monitor per-language-pair retrieval quality in production |
| Fallback | If cross-lingual confidence is low, translate query and retry in target language |
| Caching | Cache embeddings by language to avoid redundant encoding |

## How to Improve This Project

1. **Better embeddings**: Use BGE-M3 or multilingual-e5-large for stronger cross-lingual alignment, especially for CJK languages
2. **Hybrid retrieval**: Combine dense multilingual embeddings with BM25 on translated queries for higher recall
3. **Language-aware re-ranking**: Re-rank results to ensure diverse language coverage in top-K
4. **LLM answer generation**: Use Qwen3-Instruct to synthesize fluent answers from multilingual sources
5. **Real corpus**: Replace the synthetic corpus with real multilingual Wikipedia articles or news
6. **Evaluation depth**: Add per-language NDCG and per-pair MRR metrics for production monitoring

## Key Takeaways

1. **Multilingual embeddings create a shared semantic space** where meaning is preserved across languages, enabling cross-lingual retrieval without translation
2. **Cross-lingual retrieval works** -- queries in one language successfully retrieve relevant documents in other languages, though with slightly lower similarity scores
3. **Language detection** can be done via embedding similarity to reference phrases, enabling automatic response language selection
4. **Same-language bias** is a real phenomenon -- models tend to score same-language pairs higher, which should be monitored and mitigated
5. **Language pair quality varies** -- European language pairs (en-fr, en-es) typically align better than distant pairs (en-ja)
6. **Production systems** should use dedicated language detection, larger multilingual models, and LLM-based answer generation for best results